# 02 — Sample one image per metadata series

When the spreadsheet identifies several photographs as one document (same
`archief` + `fonds` + `signatuur` + calendar date), keep only the **first**
image (lowest `volgnummer`). The rest are logged, not deleted from disk.

Identity fields live on the **dorse** (`m`) row; join to the recto on
`volgnummer`. Photos with no `signatuur` are left untouched (a later hash
pass can still catch photographic twins among those).

Different dates under the same shelfmark stay separate series (e.g. Rijsel B
`247` on 1299-03-27 vs 1299-03-11).

In [ ]:
from collections import defaultdict
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import csv
import html

from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None

ROOT = Path("..").resolve()
META = ROOT / "images" / "metadata.xlsx"
RECTO = ROOT / "images" / "archive-recto"
OUT_CSV = ROOT / "data" / "series_sample.csv"
OUT_QC = ROOT / "outputs" / "stage2_series_qc.html"

NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


def col_row(ref):
    col, i = 0, 0
    while i < len(ref) and ref[i].isalpha():
        col = col * 26 + (ord(ref[i].upper()) - 64)
        i += 1
    return col - 1, int(ref[i:])


def load_xlsx(path):
    with ZipFile(path) as z:
        strings = []
        shared = ET.fromstring(z.read("xl/sharedStrings.xml"))
        for si in shared:
            strings.append("".join(
                t.text or "" for t in si.iter(
                    "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")))
        root = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in root.find("m:sheetData", NS):
            cells = {}
            for c in row:
                ref = c.get("r")
                if not ref:
                    continue
                ci, _ = col_row(ref)
                t, v = c.get("t"), c.find("m:v", NS)
                if t == "s" and v is not None:
                    val = strings[int(v.text)]
                elif v is not None:
                    val = v.text
                else:
                    val = ""
                cells[ci] = val or ""
            if cells:
                mx = max(cells)
                rows.append([cells.get(i, "") for i in range(mx + 1)])
        return rows

## Join dorse metadata onto each photo number

In [ ]:
header, *body = load_xlsx(META)
W = len(header)
raw = [dict(zip(header, (r + [""] * W)[:W])) for r in body]

by_id = defaultdict(dict)
for d in raw:
    fn = str(d.get("bestandsnaam", "")).lower()
    side = "o" if fn.endswith("o.jpg") else "m" if fn.endswith("m.jpg") else None
    if side:
        by_id[int(d["volgnummer"])][side] = d

charters = []
for pid, sides in sorted(by_id.items()):
    m = sides.get("m", {})
    charters.append({
        "volgnummer": pid,
        "recto": f"{pid}o.png",
        "archief": (m.get("archief") or "").strip(),
        "fonds": (m.get("fonds") or "").strip(),
        "signatuur": (m.get("signatuur") or "").strip(),
        "jaar": (m.get("jaar") or "").strip(),
        "maand": (m.get("maand") or "").strip(),
        "dag": (m.get("dag") or "").strip(),
        "extra_info": (m.get("extra info") or "").strip(),
    })

print(f"photo pairs: {len(charters)}")
print(f"with signatuur: {sum(1 for c in charters if c['signatuur'])}")

## Keep the first image of each series

In [ ]:
def series_key(c):
    if not c["signatuur"]:
        return None
    return (c["archief"], c["fonds"], c["signatuur"],
            c["jaar"], c["maand"], c["dag"])


groups = defaultdict(list)
for c in charters:
    k = series_key(c)
    if k:
        groups[k].append(c)

series = {k: sorted(v, key=lambda x: x["volgnummer"])
          for k, v in groups.items() if len(v) >= 2}

first_of = {}
for members in series.values():
    keep_id = members[0]["volgnummer"]
    for c in members:
        first_of[c["volgnummer"]] = keep_id

rows = []
for c in charters:
    keep_id = first_of.get(c["volgnummer"])
    if keep_id is None:
        decision, reason = "keep", ""
    elif keep_id == c["volgnummer"]:
        decision, reason = "keep", "series_first"
    else:
        decision, reason = "drop", f"series_of={keep_id}o"
    rows.append({**c, "decision": decision, "reason": reason,
                 "series_first": keep_id or ""})

n_drop = sum(1 for r in rows if r["decision"] == "drop")
print(f"series: {len(series)}")
print(f"dropped: {n_drop}  kept from those series: {len(series)}")
print(f"gallery after this stage: {len(charters) - n_drop} / {len(charters)}")
print()
for k, members in sorted(series.items(), key=lambda kv: -len(kv[1])):
    ids = [c["volgnummer"] for c in members]
    a, f, s, y, mo, d = k
    print(f"  keep {ids[0]:4d}  drop {ids[1:]}   {a} / {f} / {s}   {y}-{mo}-{d}")

In [ ]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
fields = ["volgnummer", "recto", "decision", "reason", "series_first",
          "archief", "fonds", "signatuur", "jaar", "maand", "dag", "extra_info"]
with OUT_CSV.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)
print(f"wrote {OUT_CSV}")

## QC — first of series vs the dropped frames

In [ ]:
import base64, io


def thumb_b64(path, long=280):
    im = Image.open(path)
    im = ImageOps.exif_transpose(im).convert("RGB")
    w, h = im.size
    s = long / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=70)
    return base64.b64encode(buf.getvalue()).decode()


blocks = []
for k, members in sorted(series.items(), key=lambda kv: -len(kv[1])):
    a, f, s, y, mo, d = k
    ids = [c["volgnummer"] for c in members]
    cells = []
    for i, pid in enumerate(ids):
        p = RECTO / f"{pid}o.png"
        label = "KEEP" if i == 0 else "drop"
        if p.exists():
            img = f'<img src="data:image/jpeg;base64,{thumb_b64(p)}">'
        else:
            img = "<em>missing</em>"
        cells.append(
            f'<div class="cell {label.lower()}"><div class="cap">{label} {pid}o</div>{img}</div>'
        )
    title = html.escape(f"{a} / {f} / {s}  {y}-{mo}-{d}")
    blocks.append(f"<h2>{title}</h2><p>keep {ids[0]}o · drop {', '.join(str(x)+'o' for x in ids[1:])}</p>"
                  f"<div class='row'>{''.join(cells)}</div>")

OUT_QC.parent.mkdir(parents=True, exist_ok=True)
OUT_QC.write_text(
    "<!doctype html><meta charset=utf-8><title>series sample QC</title>"
    "<style>body{font-family:system-ui;margin:24px;background:#111;color:#eee}"
    "h2{font-size:15px;margin:28px 0 4px}.row{display:flex;flex-wrap:wrap;gap:8px}"
    ".cell{width:160px}.cell img{width:160px;display:block;border-radius:4px}"
    ".cap{font:12px ui-monospace,monospace;margin-bottom:4px}"
    ".keep .cap{color:#3ddc84;font-weight:600}.drop .cap{color:#f0883e}</style>"
    f"<h1>Metadata series — keep first</h1>"
    f"<p>{len(series)} series · {n_drop} dropped · {len(charters)-n_drop} remain</p>"
    + "\n".join(blocks),
    encoding="utf-8",
)
print(f"QC sheet → {OUT_QC}")